In [ ]:
# Replay: play a fixed 10 second walk on this machine, then show the video and the timings.
# Nothing is streamed while it runs, so the numbers are the GPU's own speed. Needs a T4 GPU runtime.
import os, shutil, subprocess, sys
from IPython.display import Image, Video, display

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]  # -fmad=false: CPU and GPU match bit for bit

repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
build = os.path.join(repo, "build_replay")   # its own folder, so a running viewer is left alone


def sh(*cmd, cwd=None, quiet=False):
    """Run a command, print its output, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if not quiet or r.returncode != 0:
        print(r.stdout, flush=True)
    if r.returncode != 0:
        raise SystemExit(f"FAILED (exit {r.returncode}): {' '.join(cmd)}")
    return r.stdout


# 1. Get the code.
if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)  # what is being run
sh("nvidia-smi", "--query-gpu=name", "--format=csv,noheader")

# 2. Compile both programs. Run project once: it saves the scene file.
shutil.rmtree(build, ignore_errors=True)
os.makedirs(build)
for name in ["project", "viewer"]:
    sh("nvcc", *FLAGS, os.path.join(repo, "src", name + ".cu"), "-o", os.path.join(build, name))
out = sh("./project", cwd=build, quiet=True)
print("\n".join(line for line in out.splitlines() if "scene" in line or "differ" in line))

# 3. Play the walk and measure it.
sh(sys.executable, os.path.join(repo, "viewer", "replay.py"),
   "--renderer", os.path.join(build, "viewer"),
   "--scene", os.path.join(build, "scene.bin"),
   "--out", build)

# 4. Show the chart and the video.
chart = os.path.join(build, "walk_timing.png")
if os.path.exists(chart):
    display(Image(chart))
for name in ["walk.mp4", "walk.gif"]:
    path = os.path.join(build, name)
    if os.path.exists(path):
        display(Video(path, embed=True) if name.endswith(".mp4") else Image(path))
        break
